### Notebook scope

### Purpose
Test vertical coherence from Z300 source metrics to EPFz profiles.

### Scientific question
Is the Z300-EP100 relation part of a vertical propagation chain?

### Method
Correlate Z300 stationary projection and wave2 amplitude with EPFz at 300, 200, 150, 100, 70, and 50 hPa.

### Expected output
One selected-case profile figure and one all-case heatmap.

### Interpretation
Continuous correlation through levels supports a physical propagation chain.

### Caveat
EP-flux profiles depend on available pressure levels and no-omega-consistent EPflux_daily_ubar products.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Compute vertical EPFz correlations

### Purpose
Build profile correlations for source-diagnosable cases.

### Scientific question
Does Z300 stationary projection correlate with EPFz W1+W2 through the column?

### Method
Use source windows and EPFz = mean(-ep2), 40-80N, levels 300-50 hPa.

### Expected output
outputs/tables/06_Z300_to_EPFz_vertical_correlations_all_cases.csv and figures 06-1/06-2.

### Interpretation
Coherent sign from 300/200 to 100 hPa is stronger evidence than an isolated 100 hPa correlation.

### Caveat
A missing level or wave product becomes NA and is logged.


In [ ]:
inv = discover_hindcast_cases()
rows = []
levels = [300,200,150,100,70,50]
for case in inv.loc[inv["can_source_diagnose"], "case"]:
    window = source_windows_for_case(case)["primary"]
    z = load_or_build_z300(case, window)
    target = compute_z300_climatological_stationary_target(window)
    if z is None or target is None:
        continue
    zrows = []
    for mid in z["member"].values:
        za = compute_z300_stationary_anomaly(z.sel(member=mid))
        zrows.append({"member": member_short_id(mid), "Z300_stationary_projection": weighted_projection(za, target), "Z300_wave2_amplitude": z300_wave_amplitude(za, 2)})
    zdf = pd.DataFrame(zrows)
    profiles = []
    for wave in ["all_waves", "wave1", "wave2"]:
        profiles.append(compute_ep_vertical_profile(case, wave, window, levels))
    prof = pd.concat(profiles, ignore_index=True) if profiles else pd.DataFrame()
    if prof.empty:
        continue
    wide_prof = prof.pivot_table(index=["member", "plev_hpa"], columns="wave", values="EPFz").reset_index()
    wide_prof["wave1_plus_wave2"] = wide_prof.get("wave1", np.nan) + wide_prof.get("wave2", np.nan)
    join = wide_prof.merge(zdf, on="member", how="left")
    for plev, sub in join.groupby("plev_hpa"):
        for zcol in ["Z300_stationary_projection", "Z300_wave2_amplitude"]:
            for epcol in ["all_waves", "wave1_plus_wave2", "wave2"]:
                c = finite_corr(sub[zcol], sub[epcol])
                rows.append({"case": case, "plev_hpa": plev, "z_metric": zcol, "ep_metric": epcol, **c})
vert = pd.DataFrame(rows)
vert_csv = TAB_DIR / "06_Z300_to_EPFz_vertical_correlations_all_cases.csv"
vert.to_csv(vert_csv, index=False)
selected = [c for c in ["0008-01"] if c in set(vert.get("case", []))]
if not selected and not vert.empty:
    selected = [vert["case"].iloc[0]]
fig, axes = plt.subplots(1, max(1, len(selected)), figsize=(5.5*max(1,len(selected)), 5.8), constrained_layout=True, squeeze=False)
if vert.empty:
    axes.ravel()[0].axis("off"); axes.ravel()[0].text(0.5,0.5,"No vertical correlations",ha="center")
else:
    for ax, case in zip(axes.ravel(), selected):
        for epcol, color in [("all_waves","black"),("wave1_plus_wave2","tab:purple"),("wave2","tab:orange")]:
            sub = vert[(vert["case"]==case)&(vert["z_metric"]=="Z300_stationary_projection")&(vert["ep_metric"]==epcol)].sort_values("plev_hpa")
            ax.plot(sub["R"], sub["plev_hpa"], marker="o", color=color, label=epcol)
            sig = sub[sub["p"] < 0.05]
            ax.scatter(sig["R"], sig["plev_hpa"], s=90, facecolors="none", edgecolors=color)
        ax.set_yscale("log"); ax.invert_yaxis(); ax.set_yticks(levels); ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.axvline(0, color="0.4", lw=0.8); ax.set_xlabel("R"); ax.set_ylabel("Pressure (hPa)"); ax.set_title(case); ax.legend(fontsize=8)
savefig(fig, "06_Z300_to_EPFz_vertical_profiles_selected_cases", notebook="06_vertical_propagation_chain.ipynb", scientific_question="Does Z300 projection correlate with EPFz through multiple levels?", variables_windows="Z300 projection; EPFz all/W1+W2/wave2; 300-50 hPa; 40-80N", interpretation="Vertical continuity of R supports a propagation chain from troposphere to lower stratosphere.", caveat="Selected-case profiles are member correlations and sensitive to n=30 ensemble size.", csv_table=vert_csv)
plt.show()
mat_sub = vert[(vert["z_metric"]=="Z300_stationary_projection")&(vert["ep_metric"]=="wave1_plus_wave2")]
mat = mat_sub.pivot(index="plev_hpa", columns="case", values="R") if not mat_sub.empty else pd.DataFrame()
fig, ax = plt.subplots(figsize=(max(8, 0.45*len(mat.columns)+4), 5.2), constrained_layout=True)
if mat.empty:
    ax.axis("off"); ax.text(0.5,0.5,"No vertical heatmap data",ha="center")
else:
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_yticks(range(len(mat.index)), [f"{v:g}" for v in mat.index]); ax.set_xticks(range(len(mat.columns)), mat.columns, rotation=60, ha="right")
    for i, plev in enumerate(mat.index):
        for j, case in enumerate(mat.columns):
            p = mat_sub[(mat_sub["case"]==case)&(mat_sub["plev_hpa"]==plev)]["p"]
            star = "*" if len(p) and p.iloc[0] < 0.05 else ""
            ax.text(j,i,f"{mat.iloc[i,j]:.2f}{star}" if np.isfinite(mat.iloc[i,j]) else "NA",ha="center",va="center",fontsize=7)
    fig.colorbar(im, ax=ax, label="R")
    ax.set_title("R(Z300 projection, EPFz W1+W2) vertical coherence")
savefig(fig, "06_Z300_to_EPFz_vertical_heatmap_all_cases", notebook="06_vertical_propagation_chain.ipynb", scientific_question="Is vertical coherence robust across cases?", variables_windows="Z300 stationary projection vs EPFz W1+W2; 300-50 hPa", interpretation="Columns with same-sign correlations across levels support coherent upward wave propagation.", caveat="Gray/NA cells mean missing EPFlux levels or products.", csv_table=vert_csv)
plt.show()
write_figure_guide()